# Emotion-Aware Response Generator: Colab GPU Training

This notebook trains the GoEmotions encoder classifier and the DailyDialog mini GPT response generator on a Colab GPU.

## 1. Enable GPU

In Colab, choose `Runtime > Change runtime type > T4 GPU` or another GPU before running the training cells.

In [3]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


## 2. Get The Project Into Colab

Option A: upload the whole project folder into Colab's file browser.

Option B: put the project folder in Google Drive, then mount Drive and set `PROJECT_DIR` to that folder.

In [4]:
from pathlib import Path
print([p.name for p in Path('/content').iterdir()])


['.config', 'drive', 'sample_data']


In [3]:
from pathlib import Path
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/Final Project')
os.chdir(PROJECT_DIR)

print('working directory:', os.getcwd())
print('files:', os.listdir('.')[:20])


Mounted at /content/drive
working directory: /content/drive/MyDrive/Final Project
files: ['requirements.txt', 'chat_with_models.py', 'README.md', 'COLAB.md', 'Emotion_Response_Colab.ipynb', 'checkpoints', 'emotion_response']


In [6]:
# # Option B: uncomment these lines if your project is in Google Drive.
# # from google.colab import drive
# # drive.mount('/content/drive')
# # PROJECT_DIR = '/content/drive/MyDrive/Final Project'

# # Option A default: upload this project folder to /content/Final Project.
# PROJECT_DIR = '/content/Final Project'

# import os
# os.chdir(PROJECT_DIR)
# print('working directory:', os.getcwd())
# print('files:', os.listdir('.')[:20])

## 3. Install Dependencies

In [4]:
!pip install -q -r requirements.txt

import datasets, tqdm
print('datasets:', datasets.__version__)

datasets: 4.0.0


## 4. Train Emotion Classifier

This trains the encoder-only classifier on GoEmotions and saves validation metrics.

In [6]:
!python -m emotion_response.train_emotion_classifier --resume --epochs 12 --batch-size 128 --max-len 96 --max-vocab 30000 --d-model 192 --nhead 6 --layers 3 --class-weights inverse_sqrt --out checkpoints/emotion.pt --metrics-out checkpoints/emotion_metrics.json

README.md: 9.40kB [00:00, 25.1MB/s]
simplified/train-00000-of-00001.parquet: 100% 2.77M/2.77M [00:01<00:00, 2.71MB/s]
simplified/validation-00000-of-00001.par(…): 100% 350k/350k [00:00<00:00, 1.63MB/s]
simplified/test-00000-of-00001.parquet: 100% 347k/347k [00:00<00:00, 562kB/s] 
Generating train split: 100% 43410/43410 [00:00<00:00, 609555.16 examples/s]
Generating validation split: 100% 5426/5426 [00:00<00:00, 666909.70 examples/s]
Generating test split: 100% 5427/5427 [00:00<00:00, 549579.60 examples/s]
epoch 1: 340it [00:20, 16.48it/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = tor

## 5. Train Emotion-Conditioned Generator

This trains the mini GPT-style model on DailyDialog with prompts like `emotion: sadness user: ... response:`.

In [5]:
!python -m emotion_response.train_response_generator --resume --balance-emotions --epochs 40 --batch-size 128 --max-len 128 --max-vocab 50000 --d-model 256 --nhead 8 --layers 6 --out checkpoints/generator.pt

README.md: 7.27kB [00:00, 18.0MB/s]
daily_dialog.py: 4.85kB [00:00, 14.6MB/s]
README.md: 100% 892/892 [00:00<00:00, 5.85MB/s]
data/train-00000-of-00001-f151c79abb2c1f(…): 100% 3.61M/3.61M [00:05<00:00, 678kB/s] 
data/validation-00000-of-00001-2407eb323(…): 100% 334k/334k [00:00<00:00, 546kB/s] 
data/test-00000-of-00001-66dc7d981b70c91(…): 100% 331k/331k [00:00<00:00, 796kB/s] 
Generating train split: 100% 11118/11118 [00:00<00:00, 118841.34 examples/s]
Generating validation split: 100% 1000/1000 [00:00<00:00, 125913.48 examples/s]
Generating test split: 100% 1000/1000 [00:00<00:00, 136809.45 examples/s]
resuming from epoch 37
epoch 38: 3411it [21:45,  2.61it/s]
epoch=38 train_loss=0.2979 val_loss=3.8115
epoch 39: 3411it [21:49,  2.61it/s]
epoch=39 train_loss=0.2944 val_loss=3.8222
epoch 40: 3411it [21:48,  2.61it/s]
epoch=40 train_loss=0.2914 val_loss=3.8570


## 6. Train Input-Only Generator

This trains a baseline response model that sees only `user: {message} response:` with no emotion label.

In [ ]:
#!python -m emotion_response.train_input_response_generator --resume --epochs 40 --batch-size 128 --max-len 128 --max-vocab 50000 --d-model 256 --nhead 8 --layers 6 --out checkpoints/input_generator.pt

epoch 1: 42it [00:14,  2.95it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/drive/MyDrive/Final Project/emotion_response/train_input_response_generator.py", line 171, in <module>
    main()
  File "/content/drive/MyDrive/Final Project/emotion_response/train_input_response_generator.py", line 159, in main
    total_loss += loss.item()
                  ^^^^^^^^^^^
KeyboardInterrupt


## 7. Test End-To-End Inference

In [ ]:
!python -m emotion_response.infer --text "I cannot believe you forgot my birthday." --classifier checkpoints/emotion.pt --generator checkpoints/generator.pt --input-generator checkpoints/input_generator.pt --top-k 20 --temperature 0.75 --seed 7

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
{
  "input": "I cannot believe you forgot my birthday.",
  "emotion_classification_before_mapping": {
    "label": "neutral",
    "confidence": 0.889767050743103,
    "source": "checkpoint"
  },
  "mapped_daily_dialog_emotion": "neutral",
  "generator_conditioning_prompt": "emotion: neutral user: I cannot believe you forgot my birthday. response:",
  "sample_responses": [
    {
      "response": "i believe i will soon understand her.",
      "why_it_corresponds_to_emotion": "The input was classified as neutral, 

## 8. Zip And Download Checkpoints

In [6]:
!zip -r emotion_response_checkpoints.zip checkpoints

try:
    from google.colab import files
    files.download('emotion_response_checkpoints.zip')
except Exception as exc:
    print('Download manually from the Colab file browser:', exc)

  adding: checkpoints/ (stored 0%)
  adding: checkpoints/emotion_smoke_metrics.json (deflated 95%)
  adding: checkpoints/emotion_smoke.pt (deflated 9%)
  adding: checkpoints/generator_smoke.pt (deflated 8%)
  adding: checkpoints/emotion_metrics.json (deflated 84%)
  adding: checkpoints/emotion.pt (deflated 8%)
  adding: checkpoints/generator.pt (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: Copy Checkpoints To Google Drive

In [ ]:
# Uncomment if Google Drive is mounted.
# !mkdir -p /content/drive/MyDrive/emotion_response_checkpoints
# !cp checkpoints/emotion.pt checkpoints/emotion_metrics.json checkpoints/generator.pt checkpoints/input_generator.pt /content/drive/MyDrive/emotion_response_checkpoints/